<a href="https://colab.research.google.com/github/aryakamat01-sketch/pbs-generic-erosion/blob/main/pbs_generic_erosion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
print(pd.__version__)

2.2.3


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Data

PBS and RPBS Date of Supply supplementary reports, July 2022 – June 2026
(48 months). Source: Australian Government Department of Health, Disability
and Ageing, www.pbs.gov.au

These supplementary reports include dispensing pharmacy type but exclude
Section 100 special arrangements, so this analysis covers general community
supply only.

In [3]:
import glob

files = sorted(glob.glob('/content/drive/MyDrive/pbs-data/dos-*.csv'))
print(files)

frames = [pd.read_csv(f) for f in files]
raw = pd.concat(frames, ignore_index=True)
print(raw.shape)

['/content/drive/MyDrive/pbs-data/dos-jul-2022-to-jun-2023-phrmcy-type.csv', '/content/drive/MyDrive/pbs-data/dos-jul-2023-to-jun-2024-phrmcy-type.csv', '/content/drive/MyDrive/pbs-data/dos-jul-2024-to-jun-2025-phrmcy-type.csv', '/content/drive/MyDrive/pbs-data/dos-jul-2025-to-jun-2026-phrmcy-type.csv']
(1490524, 12)


## Joining drug names

The supplementary reports contain ITEM_CODE but no drug name, so the PBS
item-to-drug mapping file is required. The file is not UTF-8 encoded and
needs Latin-1.

192 of 789,577 rows found no matching drug name (0.02%) — most likely
historical item codes or the 99999Z placeholder used for unlisted RPBS items.
Left as-is rather than dropped.

In [4]:
drug_map = pd.read_csv('/content/drive/MyDrive/pbs-data/pbs-item-drug-map.csv', encoding='latin-1')
print(drug_map.shape)
drug_map.head()

(12162, 4)


,ITEM_CODE,DRUG_NAME,FORM/STRENGTH,ATC5_Code
0,00000A,MISSING ITEM CODE,Missing Item Code,Z
1,00013Q,EXTEMPORANEOUSLY PREPARED,Creams,Z
2,00015T,EXTEMPORANEOUSLY PREPARED,Ear drops,Z
3,00016W,ELIXIRS,Generic term,Z
4,00019B,EXTEMPORANEOUSLY PREPARED,Eye drops containing cocaine hcl,Z


## Scope decisions

**Above co-payment only.** Under co-payment scripts record a zero government
contribution, so total cost is zero and cost per script would compute as $0.
Including them would fabricate price collapses that never happened.
This excludes roughly 16% of rows.

**S90 community pharmacies only.** Excludes public and private hospital
dispensing (S94) and approved medical practitioners (S92), where pricing
arrangements differ. Retail generic competition is the question here, so
community pharmacy is the right population.

In [5]:
work = raw[(raw['SCRIPT_TYPE'] == 'ABOVE CO-PAYMENT') & (raw['PHRMCY_TYPE'] == 'S90')].copy()
work = work.merge(drug_map, on='ITEM_CODE', how='left')

print(work.shape)
print(work['DRUG_NAME'].isna().sum())

(789577, 15)
192


## Key metric: cost per script

cost_per_script = TOTAL_COST / PRESCRIPTIONS

**Important limitation.** TOTAL_COST includes the ex-manufacturer price plus
wholesale and retail mark-up, the dispensing fee and other fees. The
dispensing fee is roughly fixed per script, which puts a floor under cost per
script — the observed minimum across the dataset is $6.19.

This means measured price erosion systematically **understates** true
manufacturer price erosion, and the understatement is worst for the cheapest
drugs. A generic whose manufacturer price falls 90% may show far less
erosion here.

TOTAL_COST also excludes brand premiums paid by patients, so any premium
an originator charges after generic entry is invisible in this data.

In [6]:
work['date'] = pd.to_datetime(work['MONTH_OF_SUPPLY'], format='%Y%m')
work['cost_per_script'] = work['TOTAL_COST'] / work['PRESCRIPTIONS']

print(work['date'].min(), work['date'].max())
print(work['date'].nunique())
print(work['cost_per_script'].describe())

2022-07-01 00:00:00 2026-06-01 00:00:00
48
count    789577.000000
mean        569.329926
std        1939.327600
min           6.190000
25%          22.560000
50%          45.680000
75%         225.980000
max      201042.960000
Name: cost_per_script, dtype: float64


## Completeness check

The Explanatory Notes warn that recent months are less complete and that the
report "should not be used to analyse the recent trends". Checking total
prescriptions for the final six months: values range 17.3M–20.0M with the
last month among the highest, showing no sign of truncation. All 48 months
retained.

In [7]:
work.groupby('date')['PRESCRIPTIONS'].sum().tail(6)

,PRESCRIPTIONS
date,
2026-01-01,17255493
2026-02-01,17553355
2026-03-01,19947341
2026-04-01,18935944
2026-05-01,19676963
2026-06-01,19992133
